# 04 - Risk Scoring Model

- Baseline: Logistic Regression tren bien WOE
- So sanh: LightGBM / XGBoost (+ SHAP)
- Danh gia: AUC-ROC, KS, Gini
- Chon model chinh, luu artifact vao models/

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.paths import DATA_RAW, DATA_INTERIM, DATA_PROCESSED, MODELS, REPORTS, REPORTS_FIGURES

## Baseline: Logistic Regression + WOE

In [2]:
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from src.data.filter_vintage import assert_no_leakage

train_df = pd.read_parquet(DATA_PROCESSED / 'train.parquet')
val_df = pd.read_parquet(DATA_PROCESSED / 'val.parquet')
test_df = pd.read_parquet(DATA_PROCESSED / 'test.parquet')

WOE_FEATURES = [c for c in train_df.columns if c.endswith('_woe')]
RAW_FEATURES = [c[:-4] for c in WOE_FEATURES]  # ten bien goc (chua WOE-transform), dung cho GBM
assert_no_leakage(RAW_FEATURES)  # lop chan cuoi truoc khi train
print('WOE features:', WOE_FEATURES)

X_train, y_train = train_df[WOE_FEATURES], train_df['bad_flag']
X_val, y_val = val_df[WOE_FEATURES], val_df['bad_flag']
X_test, y_test = test_df[WOE_FEATURES], test_df['bad_flag']

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_train_auc = roc_auc_score(y_train, lr.predict_proba(X_train)[:, 1])
lr_val_auc = roc_auc_score(y_val, lr.predict_proba(X_val)[:, 1])
lr_test_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f'LR AUC - train: {lr_train_auc:.4f}, val: {lr_val_auc:.4f}, test: {lr_test_auc:.4f}')

# --- Kiem tra dau he so ---
# Quy uoc WOE cua optbinning: WoE = ln(P(x|good) / P(x|bad)), nen bin IT rui ro co WoE DUONG
# (vd revol_util < 17.65: bad rate 12.7% < 16.3% tong the -> WoE = +0.29).
# Model du bao bad_flag = 1, nen he so ky vong AM cho MOI bien: WOE cao -> rui ro thap -> P(bad) thap.
# He so DUONG la bat thuong, can kiem tra lai binning/tuong quan giua cac bien.
#
# LUU Y: ban dau cell nay ghi nguoc ("ky vong tat ca he so > 0") nen no gan co 7 he so DUNG la
# co van de va bo sot dung 1 he so THUC SU sai (revol_util +0.228). Xem sprint_1_review.md R4.
coef_table = pd.DataFrame({'feature': WOE_FEATURES, 'coef': lr.coef_[0]}).sort_values('coef')
wrong_sign = coef_table.loc[coef_table['coef'] > 0, 'feature'].tolist()
if wrong_sign:
    print(f'\nCANH BAO - {len(wrong_sign)}/{len(WOE_FEATURES)} he so DUONG (sai dau): {wrong_sign}')
    print('  -> Kiem tra binning va tuong quan voi cac bien khac truoc khi dung lam scorecard.')
else:
    print(f'\nOK - toan bo {len(WOE_FEATURES)} he so deu AM, dung quy uoc WOE, scorecard dien giai duoc.')
coef_table


WOE features: ['fico_mid_woe', 'acc_open_past_24mths_woe', 'bc_open_to_buy_woe', 'num_tl_op_past_12m_woe', 'total_bc_limit_woe', 'tot_hi_cred_lim_woe', 'avg_cur_bal_woe', 'dti_woe', 'total_rev_hi_lim_woe', 'verification_status_woe', 'annual_inc_woe', 'tot_cur_bal_woe', 'mort_acc_woe', 'inq_last_6mths_woe', 'mths_since_recent_inq_woe', 'mo_sin_rcnt_tl_woe', 'open_rv_24m_woe', 'home_ownership_woe', 'mths_since_recent_bc_woe', 'mo_sin_rcnt_rev_tl_op_woe', 'mo_sin_old_rev_tl_op_woe', 'loan_to_income_woe', 'all_util_woe', 'inq_last_12m_woe', 'max_bal_bc_woe', 'open_rv_12m_woe', 'tot_cur_bal_to_income_woe', 'open_il_12m_woe', 'open_acc_6m_woe', 'mths_since_rcnt_il_woe', 'open_il_24m_woe', 'il_util_woe', 'inq_fi_woe', 'percent_bc_gt_75_woe', 'num_actv_rev_tl_woe', 'emp_length_years_woe', 'num_rev_tl_bal_gt_0_woe', 'credit_history_length_woe', 'bc_util_woe', 'revol_bal_woe']


LR AUC - train: 0.6850, val: 0.6677, test: 0.6739

CANH BAO - 9/40 he so DUONG (sai dau): ['open_rv_12m_woe', 'revol_bal_woe', 'open_acc_6m_woe', 'inq_last_12m_woe', 'open_il_24m_woe', 'avg_cur_bal_woe', 'tot_cur_bal_woe', 'mo_sin_rcnt_rev_tl_op_woe', 'credit_history_length_woe']
  -> Kiem tra binning va tuong quan voi cac bien khac truoc khi dung lam scorecard.


,feature,coef
21,loan_to_income_woe,-1.040935
20,mo_sin_old_rev_tl_op_woe,-0.633175
7,dti_woe,-0.619361
5,tot_hi_cred_lim_woe,-0.610138
1,acc_open_past_24mths_woe,-0.554087
32,inq_fi_woe,-0.504741
33,percent_bc_gt_75_woe,-0.468348
18,mths_since_recent_bc_woe,-0.456918
8,total_rev_hi_lim_woe,-0.427105
0,fico_mid_woe,-0.419656


## Model so sanh: LightGBM / XGBoost

In [3]:
import lightgbm as lgb

# GBM dung feature GOC (chua WOE-transform) chu khong dung ban WOE - tan dung kha nang xu ly
# phi tuyen va categorical native cua tree model, dung tinh than so sanh "scorecard truyen
# thong vs. ML hien dai" thay vi ep 2 model dung chung 1 pipeline feature engineering.
train_raw = train_df[RAW_FEATURES].copy()
val_raw = val_df[RAW_FEATURES].copy()
test_raw = test_df[RAW_FEATURES].copy()

cat_features = [c for c in RAW_FEATURES if not pd.api.types.is_numeric_dtype(train_raw[c])]
for c in cat_features:
    train_raw[c] = train_raw[c].astype('category')
    val_raw[c] = pd.Categorical(val_raw[c], categories=train_raw[c].cat.categories)
    test_raw[c] = pd.Categorical(test_raw[c], categories=train_raw[c].cat.categories)

lgb_train = lgb.Dataset(train_raw, label=y_train, categorical_feature=cat_features, free_raw_data=False)
lgb_val = lgb.Dataset(
    val_raw, label=y_val, reference=lgb_train, categorical_feature=cat_features, free_raw_data=False
)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'min_child_samples': 100,
    'verbosity': -1,
    'seed': 42,
}
gbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)

gbm_train_auc = roc_auc_score(y_train, gbm.predict(train_raw, num_iteration=gbm.best_iteration))
gbm_val_auc = roc_auc_score(y_val, gbm.predict(val_raw, num_iteration=gbm.best_iteration))
gbm_test_auc = roc_auc_score(y_test, gbm.predict(test_raw, num_iteration=gbm.best_iteration))
print(f'LightGBM AUC - train: {gbm_train_auc:.4f}, val: {gbm_val_auc:.4f}, test: {gbm_test_auc:.4f}')
print('best_iteration:', gbm.best_iteration)


C:\Users\PC\AppData\Local\Temp\ipykernel_11440\94362546.py:14: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  test_raw[c] = pd.Categorical(test_raw[c], categories=train_raw[c].cat.categories)


Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[729]	training's auc: 0.735331	valid_1's auc: 0.690177


LightGBM AUC - train: 0.7353, val: 0.6902, test: 0.7004
best_iteration: 729


In [4]:
import shap
import matplotlib.pyplot as plt

REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

shap_sample = test_raw.sample(min(5000, len(test_raw)), random_state=42)
explainer = shap.TreeExplainer(gbm)
shap_values = explainer.shap_values(shap_sample)
if isinstance(shap_values, list):  # mot so version tra ve list [class0, class1]
    shap_values = shap_values[1]

shap.summary_plot(shap_values, shap_sample, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / 'shap_feature_importance.png', dpi=100, bbox_inches='tight')
plt.close()

shap_importance = pd.DataFrame({
    'feature': shap_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_importance


C:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,mean_abs_shap
0,fico_mid,0.227776
1,loan_to_income,0.205666
2,dti,0.152020
3,acc_open_past_24mths,0.123913
4,tot_hi_cred_lim,0.089596
5,home_ownership,0.085291
6,mo_sin_old_rev_tl_op,0.081035
7,verification_status,0.078091
8,total_rev_hi_lim,0.076079
9,percent_bc_gt_75,0.071726


## Cat feature theo SHAP (bo noise), train lai

Cac bien qua duoc nguong IV don bien (>0.02) chi chung minh tin hieu *rieng le* voi `bad_flag`,
khong chung minh bien do con dong gop gi khi dat canh 39 bien khac trong model that. SHAP o tren
cho thay 6/40 bien co `mean|SHAP| < 0.01` (nho hon `fico_mid` tu 23-120 lan) - gan nhu khong duoc
LightGBM su dung du da lot qua bo loc IV. 5/6 bien nay cung nam trong nhom 9 he so LR sai dau o
tren (da cong tuyen voi bien khac) - cung co gia thuyet day la bien trung lap thong tin, khong phai
tin hieu doc lap. Cat bo va train lai ca 2 model tren tap con lai.

In [5]:
SHAP_NOISE_THRESHOLD = 0.01
noise_features = shap_importance.loc[
    shap_importance['mean_abs_shap'] < SHAP_NOISE_THRESHOLD, 'feature'
].tolist()
print(f'Cat {len(noise_features)}/{len(WOE_FEATURES)} bien co mean|SHAP| < {SHAP_NOISE_THRESHOLD}: {noise_features}')

WOE_FEATURES = [f for f in WOE_FEATURES if f[:-4] not in noise_features]
RAW_FEATURES = [f for f in RAW_FEATURES if f not in noise_features]
cat_features = [c for c in cat_features if c in RAW_FEATURES]
print(f'Con lai {len(WOE_FEATURES)} bien')

X_train, X_val, X_test = train_df[WOE_FEATURES], val_df[WOE_FEATURES], test_df[WOE_FEATURES]
train_raw, val_raw, test_raw = train_raw[RAW_FEATURES], val_raw[RAW_FEATURES], test_raw[RAW_FEATURES]

# --- Train lai Logistic Regression + WOE tren tap feature da cat ---
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_train_auc = roc_auc_score(y_train, lr.predict_proba(X_train)[:, 1])
lr_val_auc = roc_auc_score(y_val, lr.predict_proba(X_val)[:, 1])
lr_test_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f'\n[Sau khi cat] LR AUC - train: {lr_train_auc:.4f}, val: {lr_val_auc:.4f}, test: {lr_test_auc:.4f}')

coef_table = pd.DataFrame({'feature': WOE_FEATURES, 'coef': lr.coef_[0]}).sort_values('coef')
wrong_sign = coef_table.loc[coef_table['coef'] > 0, 'feature'].tolist()
print(f'[Sau khi cat] {len(wrong_sign)}/{len(WOE_FEATURES)} he so DUONG (sai dau): {wrong_sign}')

# --- Train lai LightGBM tren tap feature da cat ---
lgb_train = lgb.Dataset(train_raw, label=y_train, categorical_feature=cat_features, free_raw_data=False)
lgb_val = lgb.Dataset(
    val_raw, label=y_val, reference=lgb_train, categorical_feature=cat_features, free_raw_data=False
)
gbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)
gbm_train_auc = roc_auc_score(y_train, gbm.predict(train_raw, num_iteration=gbm.best_iteration))
gbm_val_auc = roc_auc_score(y_val, gbm.predict(val_raw, num_iteration=gbm.best_iteration))
gbm_test_auc = roc_auc_score(y_test, gbm.predict(test_raw, num_iteration=gbm.best_iteration))
print(f'[Sau khi cat] LightGBM AUC - train: {gbm_train_auc:.4f}, val: {gbm_val_auc:.4f}, test: {gbm_test_auc:.4f}')
print('best_iteration:', gbm.best_iteration)

# --- Tinh lai SHAP cho model da cat (day la ban se duoc luu vao shap_importance.csv/png) ---
shap_sample = test_raw.sample(min(5000, len(test_raw)), random_state=42)
explainer = shap.TreeExplainer(gbm)
shap_values = explainer.shap_values(shap_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap.summary_plot(shap_values, shap_sample, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / 'shap_feature_importance.png', dpi=100, bbox_inches='tight')
plt.close()

shap_importance = pd.DataFrame({
    'feature': shap_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_importance

Cat 6/40 bien co mean|SHAP| < 0.01: ['avg_cur_bal', 'tot_cur_bal', 'open_acc_6m', 'open_il_24m', 'open_rv_12m', 'open_il_12m']
Con lai 34 bien



[Sau khi cat] LR AUC - train: 0.6846, val: 0.6671, test: 0.6737
[Sau khi cat] 4/34 he so DUONG (sai dau): ['revol_bal_woe', 'inq_last_12m_woe', 'mo_sin_rcnt_rev_tl_op_woe', 'credit_history_length_woe']


Training until validation scores don't improve for 50 rounds


Did not meet early stopping. Best iteration is:
[972]	training's auc: 0.744309	valid_1's auc: 0.690121


[Sau khi cat] LightGBM AUC - train: 0.7443, val: 0.6901, test: 0.7002
best_iteration: 972


C:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,mean_abs_shap
0,fico_mid,0.224712
1,loan_to_income,0.211624
2,dti,0.151681
3,acc_open_past_24mths,0.125199
4,total_rev_hi_lim,0.094009
5,home_ownership,0.089693
6,tot_hi_cred_lim,0.085890
7,mo_sin_old_rev_tl_op,0.084057
8,percent_bc_gt_75,0.079556
9,verification_status,0.078604


## Tuning LightGBM (chua co truoc do - chi dung 1 bo param mac dinh)

Cat feature o buoc tren chi xac nhan 6 bien la noise (val/test AUC khong doi), khong giup giam
overfit cua LightGBM - train AUC con tang (0.744 vs 0.735) trong khi val/test dung yen, gap
train-val nong ro tu 0.045 len 0.054. Nguyen nhan that su la chua tung regularize model.

Random search **chi tren train + validation** (dung dung nguyen tac da ap dung xuyen suot notebook
nay: khong dung test de chon gi ca, test chi dung de bao cao 1 lan cuoi) tren cac tham so kiem soat
do phuc tap cua cay: `num_leaves`, `min_child_samples`, `feature_fraction`, `bagging_fraction`,
`reg_alpha`, `reg_lambda`, `learning_rate`. Chon bo tham so co val AUC cao nhat, uu tien nhung cung
xem gap train-val de tranh chon bo qua-fit train.

In [6]:
import random

random.seed(42)

PARAM_GRID = {
    'learning_rate': [0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [100, 200, 400],
    'feature_fraction': [0.7, 0.85, 1.0],
    'bagging_fraction': [0.7, 0.85, 1.0],
    'reg_alpha': [0.0, 0.5],
    'reg_lambda': [0.0, 0.5],
}
N_TRIALS = 15
FIXED_PARAMS = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'seed': 42, 'bagging_freq': 1}

trials = []
for i in range(N_TRIALS):
    trial_params = {k: random.choice(v) for k, v in PARAM_GRID.items()}
    trial_params.update(FIXED_PARAMS)
    model = lgb.train(
        trial_params,
        lgb_train,
        num_boost_round=1200,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)],
    )
    val_auc = model.best_score['valid_0']['auc']
    train_auc = roc_auc_score(y_train, model.predict(train_raw, num_iteration=model.best_iteration))
    trial_row = {k: trial_params[k] for k in PARAM_GRID}
    trial_row.update({'best_iteration': model.best_iteration, 'train_auc': train_auc, 'val_auc': val_auc,
                       'train_val_gap': train_auc - val_auc})
    trials.append(trial_row)
    print(f'[{i + 1}/{N_TRIALS}] val_auc={val_auc:.4f} gap={train_auc - val_auc:.4f} iters={model.best_iteration}')

trials_df = pd.DataFrame(trials).sort_values('val_auc', ascending=False).reset_index(drop=True)
trials_df.to_csv(REPORTS_FIGURES / 'lightgbm_tuning_trials.csv', index=False)

# Lay tung COT rieng (trials_df[k].iloc[0]) de giu dung dtype goc cua cot do (vd num_leaves
# van la int64). Lay ca hang bang .loc[0, [...]] se ep moi gia tri ve chung 1 dtype (float),
# bien num_leaves=15 thanh 15.0 - LightGBM tu choi vi doi int.
best_params = {}
for k in PARAM_GRID:
    v = trials_df[k].iloc[0]
    best_params[k] = v.item() if hasattr(v, 'item') else v
print('\nBest params (theo val AUC, trong so ' + str(N_TRIALS) + ' lan thu):', best_params)
trials_df

[1/15] val_auc=0.6913 gap=0.0437 iters=1003


[2/15] val_auc=0.6906 gap=0.0631 iters=825


[3/15] val_auc=0.6898 gap=0.0275 iters=1196


[4/15] val_auc=0.6917 gap=0.0633 iters=377


[5/15] val_auc=0.6902 gap=0.0504 iters=568


...

[12/15] val_auc=0.6896 gap=0.0729 iters=402


[13/15] val_auc=0.6911 gap=0.0570 iters=1124


[14/15] val_auc=0.6895 gap=0.0282 iters=1196


[15/15] val_auc=0.6928 gap=0.0589 iters=888

Best params (theo val AUC, trong so 15 lan thu): {'learning_rate': 0.05, 'num_leaves': 15, 'min_child_samples': 400, 'feature_fraction': 0.7, 'bagging_fraction': 0.7, 'reg_alpha': 0.0, 'reg_lambda': 0.5}


,learning_rate,num_leaves,min_child_samples,feature_fraction,bagging_fraction,reg_alpha,reg_lambda,best_iteration,train_auc,val_auc,train_val_gap
0,0.05,15,400,0.70,0.70,0.0,0.5,947,0.729148,0.692918,0.036229
1,0.02,63,400,0.85,0.70,0.5,0.5,888,0.751700,0.692774,0.058927
2,0.02,31,200,1.00,0.85,0.0,0.5,1193,0.737555,0.691851,0.045703
3,0.05,63,400,1.00,0.85,0.0,0.5,377,0.755002,0.691745,0.063257
4,0.05,15,100,1.00,0.85,0.0,0.0,1003,0.735002,0.691277,0.043725
5,0.03,15,100,0.85,0.70,0.0,0.5,1118,0.724131,0.691109,0.033023
6,0.03,31,200,1.00,1.00,0.0,0.5,1124,0.748077,0.691097,0.056980
7,0.02,31,200,0.70,1.00,0.5,0.0,1198,0.735443,0.690982,0.044461
8,0.05,15,200,0.70,1.00,0.5,0.5,1118,0.733903,0.690672,0.043231
9,0.02,63,100,1.00,1.00,0.0,0.5,825,0.753634,0.690558,0.063076


In [7]:
import json

# Train lai LightGBM voi best_params tim duoc (day la ban se duoc danh gia/luu lam model chinh)
tuned_params = {**best_params, **FIXED_PARAMS}

gbm_before_tuning_train_auc, gbm_before_tuning_val_auc = gbm_train_auc, gbm_val_auc

gbm = lgb.train(
    tuned_params,
    lgb_train,
    num_boost_round=1200,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(40), lgb.log_evaluation(0)],
)

gbm_train_auc = roc_auc_score(y_train, gbm.predict(train_raw, num_iteration=gbm.best_iteration))
gbm_val_auc = roc_auc_score(y_val, gbm.predict(val_raw, num_iteration=gbm.best_iteration))
gbm_test_auc = roc_auc_score(y_test, gbm.predict(test_raw, num_iteration=gbm.best_iteration))
print(f'[Sau tuning] LightGBM AUC - train: {gbm_train_auc:.4f}, val: {gbm_val_auc:.4f}, test: {gbm_test_auc:.4f}')
print(f'[Sau tuning] Gap train-val: {gbm_train_auc - gbm_val_auc:.4f} '
      f'(truoc tuning: {gbm_before_tuning_train_auc - gbm_before_tuning_val_auc:.4f})')
print('best_iteration:', gbm.best_iteration)
print('tuned_params:', tuned_params)

with open(REPORTS_FIGURES / 'lightgbm_tuned_params.json', 'w') as f:
    json.dump(tuned_params, f, indent=2)

# --- SHAP cho model da tune (ban se duoc luu vao shap_importance.csv/png) ---
shap_sample = test_raw.sample(min(5000, len(test_raw)), random_state=42)
explainer = shap.TreeExplainer(gbm)
shap_values = explainer.shap_values(shap_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap.summary_plot(shap_values, shap_sample, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / 'shap_feature_importance.png', dpi=100, bbox_inches='tight')
plt.close()

shap_importance = pd.DataFrame({
    'feature': shap_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_importance

Training until validation scores don't improve for 40 rounds

Early stopping, best iteration is:
[947]	training's auc: 0.729148	valid_1's auc: 0.692918


[Sau tuning] LightGBM AUC - train: 0.7291, val: 0.6929, test: 0.7023
[Sau tuning] Gap train-val: 0.0362 (truoc tuning: 0.0542)
best_iteration: 947
tuned_params: {'learning_rate': 0.05, 'num_leaves': 15, 'min_child_samples': 400, 'feature_fraction': 0.7, 'bagging_fraction': 0.7, 'reg_alpha': 0.0, 'reg_lambda': 0.5, 'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'seed': 42, 'bagging_freq': 1}


C:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,mean_abs_shap
0,loan_to_income,0.219400
1,fico_mid,0.204321
2,dti,0.157981
3,acc_open_past_24mths,0.126184
4,tot_hi_cred_lim,0.108253
5,total_rev_hi_lim,0.106807
6,mo_sin_old_rev_tl_op,0.092935
7,percent_bc_gt_75,0.091831
8,home_ownership,0.088163
9,verification_status,0.075016


## Danh gia (AUC, KS, Gini) va lua chon model

In [8]:
def ks_statistic(y_true, y_score) -> float:
    order = pd.DataFrame({'y': y_true, 'score': y_score}).sort_values('score', ascending=False)
    cum_bad = (order['y'] == 1).cumsum() / max((order['y'] == 1).sum(), 1)
    cum_good = (order['y'] == 0).cumsum() / max((order['y'] == 0).sum(), 1)
    return float((cum_bad - cum_good).abs().max())


def gini(auc: float) -> float:
    return 2 * auc - 1


lr_test_pred = lr.predict_proba(X_test)[:, 1]
gbm_test_pred = gbm.predict(test_raw, num_iteration=gbm.best_iteration)

results = pd.DataFrame({
    'model': ['Logistic Regression + WOE', 'LightGBM'],
    'auc_test': [lr_test_auc, gbm_test_auc],
    'ks_test': [ks_statistic(y_test.values, lr_test_pred), ks_statistic(y_test.values, gbm_test_pred)],
})
results['gini_test'] = gini(results['auc_test'])
results['auc_pass_0.68'] = results['auc_test'] >= 0.68
results['ks_pass_0.25'] = results['ks_test'] >= 0.25
print(results.to_string(index=False))

# On dinh theo thoi gian trong tap test (vintage effect o muc model performance, khong chi bad rate)
test_periods = test_df['issue_d'].dt.to_period('Q')
stability = pd.DataFrame({'period': test_periods, 'y': y_test.values, 'lr_pred': lr_test_pred, 'gbm_pred': gbm_test_pred})
stability_auc = stability.groupby('period').apply(
    lambda g: pd.Series({
        'n': len(g),
        'bad_rate': g['y'].mean(),
        'lr_auc': roc_auc_score(g['y'], g['lr_pred']) if g['y'].nunique() > 1 else float('nan'),
        'gbm_auc': roc_auc_score(g['y'], g['gbm_pred']) if g['y'].nunique() > 1 else float('nan'),
    }),
    include_groups=False,
)
print('\nOn dinh AUC theo quy trong tap test:')
stability_auc


                    model  auc_test  ks_test  gini_test  auc_pass_0.68  ks_pass_0.25
Logistic Regression + WOE  0.673705 0.251961   0.347409          False          True
                 LightGBM  0.702329 0.292627   0.404657           True          True

On dinh AUC theo quy trong tap test:


,n,bad_rate,lr_auc,gbm_auc
period,,,,
2017Q1,13240.0,0.197205,0.676218,0.700506
2017Q2,34120.0,0.209408,0.680377,0.704402
2017Q3,32962.0,0.205267,0.673947,0.702585
2017Q4,24882.0,0.179166,0.657902,0.697835


## Chọn model chính & lưu artifact

In [9]:
# Quy tac chon model chinh: neu LightGBM khong vuot Logistic Regression + WOE qua 0.02 AUC,
# uu tien LR vi de giai thich hon, chuan ngach credit risk (scorecard, de audit/tuan thu).
# Neu vuot ro ret, chon LightGBM va dung SHAP (o tren) de bu dap kha nang giai thich.
#
# QUAN TRONG: so sanh tren VALIDATION AUC, khong dung TEST AUC de quyet dinh - neu dung test de
# chon model roi lai dung chinh test de bao cao hieu nang, con so AUC/KS/Gini tren test se khong
# con la uoc luong khong thien lech (unbiased) cho model da chon. Test chi dung de bao cao 1 lan.
AUC_GAP_THRESHOLD = 0.02
auc_gap = gbm_val_auc - lr_val_auc
test_auc_gap = gbm_test_auc - lr_test_auc  # chi de doi chieu, KHONG dung de quyet dinh
primary_model_name = 'LightGBM' if auc_gap >= AUC_GAP_THRESHOLD else 'Logistic Regression + WOE'
print(f'AUC gap tren VALIDATION (LightGBM - LR) = {auc_gap:.4f} (nguong {AUC_GAP_THRESHOLD}) -> dung de quyet dinh')
print(f'AUC gap tren TEST (doi chieu, khong dung de quyet dinh) = {test_auc_gap:.4f}')
print(f'-> Model chinh de xuat: {primary_model_name}')

MODELS.mkdir(parents=True, exist_ok=True)
with open(MODELS / 'logistic_regression_woe.pkl', 'wb') as f:
    pickle.dump(lr, f)
gbm.save_model(str(MODELS / 'lightgbm_model.txt'))

with open(MODELS / 'primary_model.txt', 'w') as f:
    f.write(primary_model_name)

results.to_csv(REPORTS_FIGURES / 'model_comparison.csv', index=False)
stability_auc.to_csv(REPORTS_FIGURES / 'model_stability_by_quarter.csv')
shap_importance.to_csv(REPORTS_FIGURES / 'shap_importance.csv', index=False)

print('\nDa luu: models/logistic_regression_woe.pkl, models/lightgbm_model.txt, models/primary_model.txt')
print('Da luu: reports/figures/model_comparison.csv, model_stability_by_quarter.csv, shap_importance.csv')

AUC gap tren VALIDATION (LightGBM - LR) = 0.0258 (nguong 0.02) -> dung de quyet dinh
AUC gap tren TEST (doi chieu, khong dung de quyet dinh) = 0.0286
-> Model chinh de xuat: LightGBM

Da luu: models/logistic_regression_woe.pkl, models/lightgbm_model.txt, models/primary_model.txt
Da luu: reports/figures/model_comparison.csv, model_stability_by_quarter.csv, shap_importance.csv


## Run log (lịch sử các lần train)

Mỗi lần notebook này chạy xong, kết quả (AUC/KS/Gini, số dòng train/val/test, commit git, model chính) được **append** một dòng vào `reports/model_run_log.csv` — không ghi đè — để truy vết được lịch sử các lần train, khác với `model_comparison.csv` ở trên vốn chỉ phản ánh lần chạy gần nhất.

In [10]:
import subprocess
from datetime import datetime

RUN_LOG_PATH = REPORTS / 'model_run_log.csv'

try:
    git_commit = subprocess.check_output(
        ['git', 'rev-parse', '--short', 'HEAD'], cwd=Path.cwd().parent, text=True
    ).strip()
except Exception:
    git_commit = 'unknown'

lr_ks_test = ks_statistic(y_test.values, lr_test_pred)
gbm_ks_test = ks_statistic(y_test.values, gbm_test_pred)

run_row = pd.DataFrame([{
    'run_datetime': datetime.now().isoformat(timespec='seconds'),
    'git_commit': git_commit,
    'train_n': len(train_df),
    'val_n': len(val_df),
    'test_n': len(test_df),
    'n_features': len(WOE_FEATURES),
    'lr_auc_test': lr_test_auc,
    'lr_ks_test': lr_ks_test,
    'lr_gini_test': gini(lr_test_auc),
    'gbm_auc_test': gbm_test_auc,
    'gbm_ks_test': gbm_ks_test,
    'gbm_gini_test': gini(gbm_test_auc),
    'auc_gap': auc_gap,
    'auc_gap_threshold': AUC_GAP_THRESHOLD,
    'primary_model': primary_model_name,
    'note': f'Cat {len(noise_features)} bien mean|SHAP|<{SHAP_NOISE_THRESHOLD} (noise, tu 40 con {len(WOE_FEATURES)} bien), train lai ca 2 model tren tap con lai.',
}])
run_row.to_csv(RUN_LOG_PATH, mode='a', header=not RUN_LOG_PATH.exists(), index=False)

print(f'Da append 1 dong vao reports/{RUN_LOG_PATH.name} (tong so dong hien tai: '
      f'{sum(1 for _ in open(RUN_LOG_PATH, encoding="utf-8")) - 1})')

Da append 1 dong vao reports/model_run_log.csv (tong so dong hien tai: 7)
